# 수계 전해질용 신규 리튬염 후보 탐색 (GAN / PGGAN)

In [ ]:
# !pip install -r requirements.txt   # 최초 1회
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

QUICK = True
EPOCHS = 30 if QUICK else 300
SEED = 0
OUT = Path("outputs_run"); OUT.mkdir(exist_ok=True)
print("project root:", ROOT, "| torch", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

## 0. 데이터 로딩 (입력 검증 포함)

`load_smiles_csv` 는 파일 누락 · 필수 열 누락 · 유효 분자 0개에 대해 `DataError` 로 명확히 알립니다.

In [ ]:
from src.data import load_smiles_csv, DataError, SmilesVocab, SmilesDataset

DATA = Path("data/sample_electrolytes.csv")
df = load_smiles_csv(DATA)
display(df.groupby("category").size().rename("n"))
print("target(실측) 값 보유 행:", int(df["target"].notna().sum()), "/", len(df))

# 오류 메시지 확인
for bad in ["data/없는파일.csv"]:
    try:
        load_smiles_csv(bad)
    except DataError as e:
        print("DataError:", e)

## 1. GAN / PGGAN 학습

In [ ]:
from src.train import get_args, train

EVAL = dict(n_eval=512, eval_tau=0.0, eval_seed=1234, filter_preset="anion")   # 학습 중 평가 조건 (두 모델 공통)
common = ["--data", str(DATA), "--epochs", str(EPOCHS), "--eval_every", str(max(EPOCHS // 5, 1)),
          "--augment", "10", "--seed", str(SEED)] + sum([[f"--{k}", str(v)] for k, v in EVAL.items()], [])
hist_gan   = train(get_args(["--mode", "gan",   "--out", str(OUT / "gan")]   + common))
hist_pggan = train(get_args(["--mode", "pggan", "--out", str(OUT / "pggan")] + common))

{'epoch': 6, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 918, 'lossD': -3.7759, 'lossG': 3.9698, 'validity': 0.0, 'n_valid_unique': 0, 'n_not_in_train': 0, 'n_filter_pass': 0, 'sec': 14}


{'epoch': 12, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 918, 'lossD': -2.6771, 'lossG': 12.133, 'validity': 0.0, 'n_valid_unique': 0, 'n_not_in_train': 0, 'n_filter_pass': 0, 'sec': 28}


{'epoch': 18, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 918, 'lossD': -1.4467, 'lossG': 17.0362, 'validity': 0.334, 'n_valid_unique': 31, 'n_not_in_train': 31, 'n_filter_pass': 0, 'sec': 40}


{'epoch': 24, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 918, 'lossD': -1.1551, 'lossG': 19.6152, 'validity': 0.0312, 'n_valid_unique': 12, 'n_not_in_train': 12, 'n_filter_pass': 0, 'sec': 53}


{'epoch': 30, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 918, 'lossD': -1.809, 'lossG': 19.9078, 'validity': 0.2852, 'n_valid_unique': 54, 'n_not_in_train': 54, 'n_filter_pass': 0, 'sec': 68}
molecules=107  augmented=918  vocab=29  device=cpu  mode=pggan
stage 별 완결 학습 분자 수: {'stage0_L16': 487, 'stage1_L32': 830, 'stage2_L64': 918}


{'epoch': 6, 'stage': 0, 'alpha': 1.0, 'L': 16, 'n_train_stage': 487, 'lossD': 0.1053, 'lossG': 0.1278, 'validity': 0.0098, 'n_valid_unique': 5, 'n_not_in_train': 5, 'n_filter_pass': 1, 'sec': 1}


{'epoch': 12, 'stage': 1, 'alpha': 0.2, 'L': 32, 'n_train_stage': 830, 'lossD': -2.7729, 'lossG': 1.6536, 'validity': 0.9883, 'n_valid_unique': 20, 'n_not_in_train': 20, 'n_filter_pass': 0, 'sec': 3}


{'epoch': 18, 'stage': 1, 'alpha': 1.0, 'L': 32, 'n_train_stage': 830, 'lossD': -3.8375, 'lossG': 3.6487, 'validity': 0.0527, 'n_valid_unique': 1, 'n_not_in_train': 1, 'n_filter_pass': 0, 'sec': 8}


{'epoch': 24, 'stage': 2, 'alpha': 0.6, 'L': 64, 'n_train_stage': 918, 'lossD': -4.2667, 'lossG': 4.7012, 'validity': 0.4473, 'n_valid_unique': 25, 'n_not_in_train': 25, 'n_filter_pass': 1, 'sec': 20}


{'epoch': 30, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 918, 'lossD': -2.9401, 'lossG': 6.6624, 'validity': 0.0078, 'n_valid_unique': 4, 'n_not_in_train': 4, 'n_filter_pass': 0, 'sec': 34}


In [ ]:
# stage 별 학습 분자 수 / 저장된 설정 확인
cfg_pg = json.load(open(OUT / "pggan/config.json"))
print("PGGAN stage 별 완결 학습 분자 수:", cfg_pg["stage_train_counts"])
print("평가 조건:", cfg_pg["eval_config"], "| 학습 seed:", cfg_pg["train_args"]["seed"])

### 1-1. 동일 조건 비교 (같은 z · 같은 온도 · 같은 샘플 수)

In [ ]:
from src.evaluate import EvalConfig, eval_latents, evaluate_model, save_eval
from src.train import load_checkpoint

ecfg = EvalConfig(n_samples=EVAL["n_eval"], tau=EVAL["eval_tau"], seed=EVAL["eval_seed"], filter_preset="anion")
train_set = set(df["smiles"])
rows = {}
for name in ("gan", "pggan"):
    G, D, vocab, args = load_checkpoint(OUT / name / "checkpoint.pt")
    z_eval = eval_latents(ecfg, args.latent_dim)          # 두 모델에 동일한 z
    r = evaluate_model(G, vocab, ecfg, train_set, z=z_eval)
    save_eval(r, OUT / name / "final_eval.json")
    rows[name] = {k: r[k] for k in ("n_samples", "tau", "eval_seed", "validity", "n_valid_unique", "n_not_in_train", "n_filter_pass")}
cmp = pd.DataFrame(rows).T
display(cmp)
BEST = cmp.sort_values(["n_filter_pass", "n_valid_unique"], ascending=False).index[0]
print("선택된 모델 (필터 통과 수 → 유효 고유 구조 수 순):", BEST)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for name, h in [("GAN", hist_gan), ("PGGAN", hist_pggan)]:
    hd = pd.DataFrame(h)
    for ax, k in zip(axes, ["lossD", "validity", "n_valid_unique"]):
        ax.plot(hd["epoch"], hd[k], marker="o", label=name); ax.set_title(k); ax.set_xlabel("epoch")
axes[0].legend(); plt.tight_layout(); plt.show()

## 2. 후보 필터 (설정 가능, 사유 기록)

* `PRESETS["anion"]` : 단일 조각, 총 전하 −1 (리튬염 음이온 후보)
* `PRESETS["salt"]`  : 2조각(양이온+음이온), 총 전하 0, 조각 전하 (+1, −1)
* `PRESETS["neutral"]` : 단일 조각, 전하 0 (용매·첨가제)

In [ ]:
from src.filters import FilterConfig, PRESETS, PROTIC_RULES, apply_filters, filter_summary

print("Protic 구조 규칙 (표시용):", PROTIC_RULES)
demo = ["[N-]", "[P-]", "FC(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F", "CCCCCCCC", "[Li+].FS(=O)(=O)[N-]S(=O)(=O)F",
        "C[CH2]", "OCCO", "not_a_smiles", "OCCO", "OC(=O)C(F)(F)F"]
for preset in ("anion", "salt", "neutral"):
    print(f"\n=== preset: {preset} — {PRESETS[preset].note}")
    display(apply_filters(demo, PRESETS[preset], train_set)[["smiles", "passed", "reasons", "is_protic_rule", "protic_rules_matched"]])

# 사용자 정의 예: 라디칼 허용 + 최소 중원자 5
custom = FilterConfig(name="anion_custom", allowed_total_charge=(-1,), n_fragments=(1,), min_heavy_atoms=5, allow_radicals=True)
print(custom.to_dict())

## 3. 후보 생성 → 필터 → 벡터 공간 매핑 → 점수 → 선정

점수 열 이름이 실측 학습 여부를 나타냅니다: `heuristic_exploration_score` (score_type = `heuristic_proxy`, 탐색용 휴리스틱) 또는 `ml_pred_target` (실측 학습).
후보 생성 조건(seed, n, tau, sampler, 필터)은 `generation_config.json` 에 저장되며 학습 중 평가 조건과 별개입니다.

후보가 0개이거나 요청 수보다 적어도 처리는 중단되지 않고 `STATUS.json` 에 상태 코드를 남깁니다.

| 코드 | 의미 |
|---|---|
| `NO_VALID_STRUCTURES` | 생성 결과에 유효한 SMILES 가 없음 |
| `NO_FILTER_PASS` | 유효 구조는 있으나 필터 통과 구조가 없음 |
| `NO_EMBEDDABLE` | 필터는 통과했으나 임베딩 조건(모델 vocab 외 토큰 / max_len 초과)으로 제외됨 — **필터 탈락과 구분** |
| `OK_PARTIAL` | 후보가 요청 수보다 적음 |
| `OK` | 요청 수만큼 선정 |

In [ ]:
from src.candidates import run as select_candidates

GEN = dict(n=4096, top=20, tau=0.6, seed=SEED)      # 최종 후보 생성 조건 (학습 평가 조건과 별개로 명시)
results = {}
for preset in ("anion", "neutral"):
    print(f"\n##### filter preset = {preset}")
    results[preset] = select_candidates(str(OUT / BEST / "checkpoint.pt"), str(DATA), str(OUT / f"candidates_{preset}"),
                                        filter_preset=preset, region_categories=("anion",), n_clusters=5, **GEN)
    st = json.load(open(OUT / f"candidates_{preset}/STATUS.json"))
    print("STATUS:", st["code"], "-", st["message"])
    print({k: st[k] for k in ("n_valid", "n_valid_unique", "n_filter_pass", "n_embeddable", "n_not_embeddable") if k in st})
    display(pd.read_csv(OUT / f"candidates_{preset}/filter_summary.csv"))

In [ ]:
# 선정 후보표: 구조 · 선정 점수 · 점수 종류 · 필터 결과 · 선정 이유 · Protic 규칙 표시
# 구조 필터 통과는 구조 조건 충족을 뜻할 뿐 합성 가능성·수계 적용 가능성 검증이 아님.
SHOW = ["smiles", "selection_score", "score_type", "filter_passed", "filter_reasons", "selection_reason",
        "cluster", "in_reference_region", "is_protic_rule", "protic_rules_matched"]
for preset, picked in results.items():
    print(f"=== {preset}: {len(picked)} candidates")
    if len(picked) == 0:
        st = json.load(open(OUT / f"candidates_{preset}/STATUS.json"))
        print("  선정된 후보 없음 —", st["code"], ":", st["message"])
        continue
    display(picked[SHOW].head(10))
    png = OUT / f"candidates_{preset}/candidates_top.png"
    if png.exists():
        display(Image(str(png)))
    else:
        print("  (구조 그림 없음)")

### 3-1. 후보 선정 수 동작 확인

`select_top` 은 항상 `min(top, 선정 가능한 고유 후보 수)` 개를 선정합니다.
`top` 이 클러스터 수보다 작으면 각 클러스터 최고 점수 후보끼리 비교하고, 그 이상이면 클러스터별로 나눈 뒤 남은 자리를 점수순으로 채웁니다.
정렬 기준은 점수 내림차순 → 클러스터 오름차순 → SMILES 오름차순이므로 동점에서도 결과가 재현됩니다.

In [ ]:
from src.candidates import select_top

pools = {k: pd.read_csv(OUT / f"candidates_{k}/generated_passed.csv") for k in results}
pools = {k: v for k, v in pools.items() if len(v) and "cluster" in v}
if not pools:
    print("선정 가능한 후보가 없어 선정 수 동작 확인을 건너뜁니다.")
else:
    key = max(pools, key=lambda k: pools[k]["cluster"].nunique())   # 클러스터가 가장 많은 풀로 확인
    src_df = pools[key]
    print(f"후보 풀('{key}'):", len(src_df), "개 / 클러스터", src_df["cluster"].nunique(), "개")
    for t in [0, 2, 5, 10 ** 6]:
        sel = select_top(src_df, t)
        print(f"top={t:>7} -> 선정 {len(sel):>3}개"
              + (f"  클러스터 {sorted(int(c) for c in sel['cluster'].unique())}"
                 f" | {sel['selection_reason'].iloc[0][:52]}" if len(sel) else ""))
    shuffled = select_top(src_df.sample(frac=1, random_state=42), 5)
    print("입력 순서를 섞어도 동일 선정:", shuffled["smiles"].tolist() == select_top(src_df, 5)["smiles"].tolist())
    for bad in (-1, 2.5):
        try:
            select_top(src_df, bad)
        except (ValueError, TypeError) as e:
            print(f"top={bad!r} ->", type(e).__name__, str(e)[:70])

# 생성 후보가 한 클러스터에 몰리는 경우 top < 클러스터 수 분기를 실제 후보로 확인할 수 없으므로,
# 아래는 해당 분기 동작만 보이기 위한 합성 입력 (실제 생성 결과 아님).
synthetic = pd.DataFrame([{"smiles": f"C{c}_{i}", "cluster": c, "selection_score": 10 - c - 0.1 * i}
                          for c in range(5) for i in range(3)])
for t in (2, 7):
    sel = select_top(synthetic, t)
    print(f"[합성 입력] 클러스터 5개, top={t} -> {len(sel)}개, 클러스터 {sorted(int(c) for c in sel['cluster'].unique())}"
          f" | {sel['selection_reason'].iloc[0][:52]}")

[합성 입력] 클러스터 5개, top=2 -> 2개, 클러스터 [0, 1] | top 2 among per-cluster best candidates (5 clusters)
[합성 입력] 클러스터 5개, top=7 -> 7개, 클러스터 [0, 1, 2, 3, 4] | per-cluster top 1 by selection_score (5 clusters)


## 4. 벡터 공간 분석 — 두 공간을 구분

| 공간 | 무엇인가 | 용도 |
|---|---|---|
| **Generator z-space** | 생성 입력 노이즈 (Sobol 샘플) | 탐색·보간. 실제 분자를 직접 투영할 수 없음 |
| **Discriminator feature space** | D 중간 레이어 임베딩 | 실제·생성 분자 공동 투영 → PCA/KMeans, 축-기능기 상관, 기준군 분포 영역 |

점선 상자는 기준군(anion) 임베딩의 PCA 평균 ± kσ 인 **기준군 분포 영역**입니다. 실제 사용 가능 영역을 뜻하지 않습니다.

In [ ]:
from src.latent_analysis import LatentSpaceAnalyzer

pick_dir = OUT / f"candidates_{list(results)[0]}"
for title, fname in [("Generator z-space 지도 (샘플링한 z 의 PCA, 유효/필터통과 표시)", "zspace_map.png"),
                     ("Discriminator feature space 지도 (실제 분자 + 필터 통과 생성 분자 + 기준군 분포 영역)", "latent_map.png")]:
    print(title)
    display(Image(str(pick_dir / fname))) if (pick_dir / fname).exists() else print("  (그림 없음 — 생성 결과 부족)")
print(json.load(open(pick_dir / "latent/reference_region.json"))["meaning"])
display(pd.read_csv(pick_dir / "latent/cluster_summary.csv").head())
print(json.dumps(json.load(open(pick_dir / "latent/axis_interpretation.json")), ensure_ascii=False, indent=2))

### 4-1. z-space 보간 — 생성 구조 · 유효성 · 기능기 변화

유효 구조가 없거나 같은 구조만 반복돼도 표와 요약이 그대로 출력됩니다.

In [ ]:
from src.latent_analysis import sobol_latents, generate, interpolation_table
from src.metrics import canonical

G, D, vocab, args = load_checkpoint(OUT / BEST / "checkpoint.pt")
z = sobol_latents(512, args.latent_dim, seed=SEED)
det = generate(G, vocab, z, tau=0.0)
ok = [i for i, s in enumerate(det) if canonical(s)]
pairs = [(z[ok[0]], z[ok[-1]]) if len(ok) >= 2 else (z[0], z[1]),   # 유효 구조 사이
         (z[2], z[3])]                                             # 임의 z 사이 (유효 구조가 없을 수 있음)
for z0, z1 in pairs:
    tbl = interpolation_table(G, vocab, z0, z1, n_steps=8, tau=0.0)
    print("요약:", tbl.attrs["summary"])
    display(tbl)

## 5. 측정 결과 피드백 재학습 (아래 실행은 예시 데이터 — 실제 측정 결과 아님)

`data/experiments_example.csv` 는 **예시 파일**입니다. 결과 폴더의 `DATA_SOURCE.txt` 에 예시 데이터임이 기록되며,
실제 측정 데이터 파일(`smiles,target`)로 바꾸면 특성 모델이 해당 측정값을 학습해 `ml_pred_target` 으로 표기됩니다.

In [ ]:
import importlib
import inspect
import src.feedback

# 수정된 파일을 다시 불러오기
importlib.invalidate_caches()
feedback_module = importlib.reload(src.feedback)

# 실제 불러온 파일과 저장 코드 확인
print("불러온 파일:", feedback_module.__file__)
source = inspect.getsource(feedback_module.main)
print("\n".join(
    line for line in source.splitlines()
    if "write_text" in line
))

# 새로 불러온 모듈의 함수를 직접 실행
feedback_module.main([
    "--experiments", "data/experiments_example.csv",
    "--ckpt", str(OUT / BEST / "checkpoint.pt"),
    "--out", str(OUT / "feedback_example"),
    "--epochs", str(max(EPOCHS // 3, 2)),
    "--eval_every", str(max(EPOCHS // 3, 2)),
] + sum([[f"--{k}", str(v)] for k, v in EVAL.items()], []))

print(
    (OUT / "feedback_example/DATA_SOURCE.txt")
    .read_text(encoding="utf-8")
)

property model retrained on 'target'  CV R2=-5.973
molecules=107  augmented=926  vocab=29  device=cpu  mode=gan
stage 별 완결 학습 분자 수: {'stage2_L64': 926}
resumed from outputs_run/gan/checkpoint.pt


{'epoch': 10, 'stage': 2, 'alpha': 1.0, 'L': 64, 'n_train_stage': 926, 'lossD': -1.8206, 'lossG': 14.3903, 'validity': 0.1348, 'n_valid_unique': 29, 'n_not_in_train': 29, 'n_filter_pass': 0, 'sec': 22}
EXAMPLE DATA — 실제 실험 결과가 아닌 예시 파일로 실행된 결과


## 6. 체크포인트 구조/vocab 일치 검사

In [ ]:
from src.train import load_checkpoint
import torch
ck = torch.load(OUT / BEST / "checkpoint.pt", map_location="cpu")
bad = dict(ck); bad["vocab"] = ck["vocab"][:-1]           # vocab 크기를 일부러 어긋나게
torch.save(bad, OUT / "bad_ckpt.pt")
for pth in [OUT / "bad_ckpt.pt", OUT / "missing.pt"]:
    try:
        load_checkpoint(pth)
    except (ValueError, FileNotFoundError) as e:
        print(type(e).__name__, "->", str(e)[:140])
print("정상 checkpoint 로드:", load_checkpoint(OUT / BEST / "checkpoint.pt")[3].mode)

## 결과물 요약

* `outputs_run/{gan,pggan}/` : checkpoint, config.json(학습 설정·평가 조건·stage별 데이터 수), history.json, final_eval.json
* `outputs_run/candidates_{anion,neutral}/` : generation_config.json, filter_results.csv, filter_summary.csv, generated_passed.csv, candidates_top.csv(+png), zspace_map.png, latent_map.png, latent/
* `outputs_run/feedback_example/` : DATA_SOURCE.txt (예시 데이터 표시), 재학습 결과

필터·선정 로직의 단위 테스트는 `python -m pytest tests -q` 로 실행합니다 (빈 입력, 전량 무효, 필터 통과 0, 임베딩 제외,
top < 클러스터 수, 후보 부족, top=0, 잘못된 top, 동점 재현성).

In [ ]:
import pandas as pd

candidates = pd.read_csv(
    OUT / "candidates_anion" / "candidates_top.csv"
)

display(candidates[[
    "smiles",
    "selection_score",
    "score_type",
    "selection_reason",
]])